In [1]:
!git clone https://github.com/Artbaj/ConfLGN.git



Cloning into 'ConfLGN'...
remote: Enumerating objects: 266, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 266 (delta 6), reused 4 (delta 2), pack-reused 248 (from 2)
Receiving objects: 100% (266/266), 242.15 MiB | 33.52 MiB/s, done.
Resolving deltas: 100% (90/90), done.
Updating files: 100% (91/91), done.


In [19]:
!pip install medmnist

!pip install -e ./torchlogix


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 6.9 MB/s eta 0:00:00
Obtaining file:///content/ConfLGN/torchlogix
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for torchlogix (pyproject.toml) ... done
  Created wheel for torchlogix: filename=torchlogix-0.1.1-0.editable-py3-none-any.whl size=5281 sha256=803f5489ca27a7496e0bde1404ad7d1fe870e656ef894616dd5d4d089e0b6a60
  Stored in directory: /tmp/pip-ephem-wheel-cache-tvy0wsih/wheels/cb/c7/52/4441a51e4bd1442fe2682ba038982e61232446cdb75ec57d20
Successfully built torchlogix
  Attempting uninstall: torchlogix
    Found existing installation: torchlogix 0.1.1
    Uninstalling torchlogix-0.1.1:
      Successfully uninstalled torchlogix-0.1.1


In [20]:


  from pathlib import Path
  import sys

  from torchvision import transforms
  from medmnist import INFO, BloodMNIST
  import numpy as np
  from MOC.utilities.LogicNet import LogicNet
  from MOC.utilities.train_model import train_model,evaluate_one_iteration
  from MOC.utilities.reproducibility import seed_everything
  transform = transforms.Compose([
      transforms.Grayscale(num_output_channels=1),  # LogicNet currently expects 1 channel
                     # LogicNet currently expects 28x28
      transforms.ToTensor(),
  ])

  target_transform = lambda y: int(np.asarray(y).item())
  num_classes = len(INFO["tissuemnist"]["label"])
  print(num_classes)
  train_dataset = BloodMNIST(
      split="train",
      transform=transform,
      target_transform=target_transform,
      download=True,
      size=28
  )
  test_dataset = BloodMNIST(
      split="test",
      transform=transform,
      target_transform=target_transform,
      download=True,
      size=28
  )


ModuleNotFoundError: No module named 'torchlogix.layers'

In [21]:
import matplotlib.pyplot as plt

idx = 129  # numer obrazka z test setu

img, label = test_dataset[idx]

plt.imshow(img.squeeze())
plt.title(f"Label: {label}")
plt.axis("off")
plt.show()

NameError: name 'test_dataset' is not defined

In [ ]:
seed_everything(67)
model = LogicNet(num_classes=num_classes,dense_num=4,base_dense_dims=[2048,1280,640,640], conv_num=3, kernel_multiplier=3, k=128,tau=20,channels=1)


In [ ]:

import torch
model.load_state_dict(
    torch.load("last.pth", map_location="cpu", weights_only=True)
)
model.eval()

LogicNet(
  (features): Sequential(
    (0): LogicConv2d(
      (tree_weights): ParameterList(
          (0): Parameter containing: [torch.float32 of size 4x128x16]
          (1): Parameter containing: [torch.float32 of size 2x128x16]
          (2): Parameter containing: [torch.float32 of size 1x128x16]
      )
      (connections): FixedConvConnections()
    )
    (1): OrPooling2d()
    (2): LogicConv2d(
      (tree_weights): ParameterList(
          (0): Parameter containing: [torch.float32 of size 4x384x16]
          (1): Parameter containing: [torch.float32 of size 2x384x16]
          (2): Parameter containing: [torch.float32 of size 1x384x16]
      )
      (connections): FixedConvConnections()
    )
    (3): OrPooling2d()
    (4): LogicConv2d(
      (tree_weights): ParameterList(
          (0): Parameter containing: [torch.float32 of size 4x1152x16]
          (1): Parameter containing: [torch.float32 of size 2x1152x16]
          (2): Parameter containing: [torch.float32 of size 1x1

In [ ]:
loss, acc = evaluate_one_iteration(
    model,
    test_dataset,
    batch_size=1,
    force_cpu=False,
)

In [ ]:
 model, history = train_model(
        model,
        train_dataset,
        test_dataset,
        lr=2e-1,
        weight_decay=0,
        batch_size=64,
        num_iterations=5000,
        metrics_every=1,
        force_cpu=False,
        save="best"
    )

cuda

iter    1 | train_loss 4.2323 | test_acc_discrete 0.3049 | test_loss_discrete 7.6289 | test_acc_relaxed 0.2078 | test_loss_relaxed 9.5749


KeyboardInterrupt: 

In [ ]:
import torch
torch.save(model.state_dict(), 'logicnet_bloodmnist.pth')

In [ ]:
!ls


experiments  logicnet_bloodmnist.pth  utilities


In [ ]:
# Interactive 3D chart for test_loss_acc. Run this after the training cell.
import numpy as np

try:
    import plotly.graph_objects as go
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Plotly is required for this interactive 3D chart. '
        'Run `%pip install plotly` in a notebook cell, restart the kernel, '
        'then run this cell again.'
    ) from exc

if 'test_loss_acc' not in globals():
    raise NameError('Run the training cell first so test_loss_acc exists.')


def _as_1d_float_array(values):
    try:
        array = np.asarray(values, dtype=float).reshape(-1)
    except (TypeError, ValueError):
        return None

    if not np.all(np.isfinite(array)):
        return None

    return array.tolist()


if not isinstance(test_loss_acc, dict):
    raise TypeError('test_loss_acc must be a dictionary of metric lists.')

metric_series = {}
for metric_name, metric_values in test_loss_acc.items():
    numeric_values = _as_1d_float_array(metric_values)
    if numeric_values is not None:
        metric_series[metric_name] = numeric_values

if not metric_series:
    raise ValueError('No numeric metric lists were found in test_loss_acc.')

lengths = {metric_name: len(values) for metric_name, values in metric_series.items()}
if len(set(lengths.values())) != 1:
    raise ValueError(f'All plottable metrics must have the same length. Lengths: {lengths}')

point_count = next(iter(lengths.values()))
if point_count == 0:
    raise ValueError('test_loss_acc has no points yet. Run training long enough to record metrics.')

metric_names = list(metric_series.keys())


def _default_metric(preferred_name, fallback_index):
    if preferred_name in metric_names:
        return preferred_name
    return metric_names[min(fallback_index, len(metric_names) - 1)]


x_default = _default_metric('train_iteration', 0)
y_default = _default_metric('test_loss_discrete', 1)
z_default = _default_metric('test_acc_discrete', 2)

hover_text = [
    '<br>'.join(
        [f'point: {point_index}']
        + [
            f'{metric_name}: {metric_series[metric_name][point_index]:.6g}'
            for metric_name in metric_names
        ]
    )
    for point_index in range(point_count)
]

fig = go.Figure(
    data=[
        go.Scatter3d(
            x=metric_series[x_default],
            y=metric_series[y_default],
            z=metric_series[z_default],
            mode='markers+lines',
            marker=dict(
                size=6,
                color=list(range(point_count)),
                colorscale='Viridis',
                showscale=True,
                colorbar=dict(title='point'),
            ),
            line=dict(width=4, color='rgba(80, 80, 80, 0.45)'),
            text=hover_text,
            hovertemplate='%{text}<extra></extra>',
        )
    ]
)


def _axis_buttons(axis_name):
    axis_title_key = f'scene.{axis_name}axis.title.text'
    return [
        dict(
            label=metric_name,
            method='update',
            args=[
                {axis_name: [metric_series[metric_name]]},
                {axis_title_key: metric_name},
            ],
        )
        for metric_name in metric_names
    ]


fig.update_layout(
    title='test_loss_acc 3D metrics',
    template='plotly_white',
    height=650,
    margin=dict(l=0, r=0, t=120, b=0),
    scene=dict(
        xaxis_title=x_default,
        yaxis_title=y_default,
        zaxis_title=z_default,
    ),
    updatemenus=[
        dict(
            buttons=_axis_buttons('x'),
            active=metric_names.index(x_default),
            direction='down',
            x=0.00,
            y=1.16,
            xanchor='left',
            yanchor='top',
        ),
        dict(
            buttons=_axis_buttons('y'),
            active=metric_names.index(y_default),
            direction='down',
            x=0.24,
            y=1.16,
            xanchor='left',
            yanchor='top',
        ),
        dict(
            buttons=_axis_buttons('z'),
            active=metric_names.index(z_default),
            direction='down',
            x=0.48,
            y=1.16,
            xanchor='left',
            yanchor='top',
        ),
    ],
    annotations=[
        dict(text='X axis', x=0.00, y=1.24, xref='paper', yref='paper', showarrow=False),
        dict(text='Y axis', x=0.24, y=1.24, xref='paper', yref='paper', showarrow=False),
        dict(text='Z axis', x=0.48, y=1.24, xref='paper', yref='paper', showarrow=False),
    ],
)

fig.show()


NameError: name 'test_loss_acc' is not defined

In [ ]:
import torch
torch.save(model.state_dict(), 'logicnet_bloodmnist.pth')